# Envelhecimento Populacional e Estado Nutricional no Brasil

## Corpus, PLN e recuperação de informação

Este notebook executa o pipeline do projeto: coleta registros bibliográficos do PubMed em três temas, constrói e deduplica um corpus, aplica pré-processamento bilíngue, representa os textos com TF-IDF, compara a busca com um baseline lexical e gera métricas, análise de erros e quatro visualizações.

> **Nota metodológica:** as categorias da avaliação são rótulos heurísticos inferidos por palavras-chave dos títulos e resumos. Não são julgamentos humanos; interprete as métricas como exploratórias.

## 1. Preparação

Instale as dependências na raiz do projeto com `python -m pip install -r requirements.txt`. A coleta usa a API pública NCBI PubMed e requer conexão à internet. O script busca até 25 registros por tema; os resultados podem variar a cada execução.

In [ ]:
from pathlib import Path
import json
import runpy
import sys

import pandas as pd
from IPython.display import Image, display

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")

## 2. Localização e coleta do corpus

O notebook procura o script a partir da pasta atual e de suas pastas superiores. Os registros são coletados do PubMed e deduplicados por PMID. IBGE, SISVAN e outras fontes permanecem planejadas, mas ainda não estão integradas a esta execução.

In [ ]:
SCRIPT_RELATIVE = Path("src/s003_envelhecimento_populacional_estado_nutricional_final.py")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / SCRIPT_RELATIVE).is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro da pasta do projeto.")
SCRIPT_PATH = ROOT / SCRIPT_RELATIVE
runpy.run_path(str(SCRIPT_PATH), run_name="__main__")

## 3. Resultados

A avaliação compara TF-IDF com baseline lexical por Precision@1, Precision@3, Recall@3 e MRR. Como o problema é ranking de busca, não se aplica matriz de confusão nesta versão. Consulte os exemplos de erros e as quatro visualizações abaixo.

In [ ]:
METADATA_PATH = ROOT / "metadados/metadata_execucao.json"
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
metrics = pd.read_csv(ROOT / "resultados/metricas_busca.csv")
manifest = pd.read_csv(ROOT / "metadados/corpus_manifest.csv")
errors = pd.read_csv(ROOT / "resultados/analise_erros.csv")

display(pd.Series(metadata, name="Valor").to_frame())
print(f"Documentos no manifesto: {len(manifest)}")
display(metrics)
print("Amostra da análise de erros:")
display(errors.head(10))

for figure_name in metadata["visualizacoes"]:
    display(Image(filename=str(ROOT / "resultados" / figure_name)))

## 4. Limitações e reprodutibilidade

- O conteúdo e a quantidade de registros variam conforme o PubMed e a data da coleta.
- Os rótulos são heurísticos e podem favorecer métodos lexicais; revise `resultados/analise_erros.csv` e faça anotação humana antes de conclusões fortes.
- Resumos completos e corpus processado são gerados localmente e ignorados pelo Git. O manifesto, as métricas agregadas e os quatro gráficos são os artefatos versionáveis.
- Para reproduzir pelo terminal, execute `python src/s003_envelhecimento_populacional_estado_nutricional_final.py`.